Salsabilla Indah Rahmawati

1. Apa nama database yang digunakan pada tutorial?

Nama database yang digunakan adalah neo4j, sebagaimana ditunjukkan oleh parameter database_="neo4j" dalam contoh driver.

2. Bagaimana mengakses database lain dalam satu instance?

with driver.session(database="neo4j") as session:
    result = session.run("MATCH (n) RETURN count(n) AS jumlah")

with driver.session(database="system") as session:
    result = session.run("SHOW DATABASES")

Database system digunakan untuk administrasi database, sedangkan kueri data biasa dijalankan pada database yang sesuai. Hak akses pengguna dan versi Neo4j menentukan kueri yang dapat digunakan.

3. Apa jenis graf pertemanan yang dibuat?

Graf berarah karena relasi KNOWS memiliki arah dari Person pertama ke Person kedua. Graf ini juga dapat memiliki relasi satu arah saja, misalnya Arthur mengetahui Guinevere belum tentu berarti Guinevere mengetahui Arthur. Relasi dibuat dengan Cypher MERGE (a)-[:KNOWS]->(friend).

In [16]:
import pandas as pd

path = r"C:\Users\ZAHRA CITRA\Downloads\archive (2)\events.csv"
events = pd.read_csv(path)

print("Timestamp minimum:", events["timestamp"].min())
print("Timestamp maksimum:", events["timestamp"].max())

events["datetime"] = pd.to_datetime(
    events["timestamp"], unit="ms", utc=True
)
print("Tanggal minimum:", events["datetime"].min())
print("Tanggal maksimum:", events["datetime"].max())

Timestamp minimum: 1430622004384
Timestamp maksimum: 1442545187788
Tanggal minimum: 2015-05-03 03:00:04.384000+00:00
Tanggal maksimum: 2015-09-18 02:59:47.788000+00:00


In [1]:
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "iyen260126."

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

try:
    driver.verify_connectivity()
    print("Koneksi ke Neo4j berhasil!")
except Exception as e:
    print("Koneksi gagal:", e)
finally:
    driver.close()

Koneksi ke Neo4j berhasil!


In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\Users\ZAHRA CITRA\Downloads\archive (2)")

events = pd.read_csv(DATA_DIR / "events.csv")
properties1 = pd.read_csv(DATA_DIR / "item_properties_part1.csv")
properties2 = pd.read_csv(DATA_DIR / "item_properties_part2.csv")
category_tree = pd.read_csv(DATA_DIR / "category_tree.csv")

print("Events:", events.shape)
print("Item properties part 1:", properties1.shape)
print("Item properties part 2:", properties2.shape)
print("Category tree:", category_tree.shape)

display(events.head())
display(properties1.head())
display(properties2.head())
display(category_tree.head())

Events: (2756101, 5)
Item properties part 1: (10999999, 4)
Item properties part 2: (9275903, 4)
Category tree: (1669, 2)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513


,timestamp,itemid,property,value
0,1433041200000,183478,561,769062
1,1439694000000,132256,976,n26.400 1135780
2,1435460400000,420307,921,1149317 1257525
3,1431831600000,403324,917,1204143
4,1435460400000,230701,521,769062


,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0


In [18]:
from pathlib import Path

OUTPUT_DIR = Path.cwd() / "hasil_ujian"
OUTPUT_DIR.mkdir(exist_ok=True)

# Contoh periode UTC; sesuaikan berdasarkan rentang data dan aturan kelas.
P1_START, P1_END = "2015-05-03", "2015-07-01"
P2_START, P2_END = "2015-07-01", "2015-09-19"

p1_start = pd.Timestamp(P1_START, tz="UTC")
p1_end = pd.Timestamp(P1_END, tz="UTC")
p2_start = pd.Timestamp(P2_START, tz="UTC")
p2_end = pd.Timestamp(P2_END, tz="UTC")

train = events[
    (events["datetime"] >= p1_start) &
    (events["datetime"] < p1_end)
].copy()

test = events[
    (events["datetime"] >= p2_start) &
    (events["datetime"] < p2_end)
].copy()

cols = [
    c for c in
    ["timestamp", "visitorid", "event", "itemid", "transactionid"]
    if c in events.columns
]

train[cols].to_csv(OUTPUT_DIR / "events_train.csv", index=False)
test[cols].to_csv(OUTPUT_DIR / "events_test.csv", index=False)

print("Jumlah training:", len(train))
print("Jumlah testing:", len(test))
print("Training:", train["datetime"].min(), "-", train["datetime"].max())
print("Testing:", test["datetime"].min(), "-", test["datetime"].max())

Jumlah training: 1201045
Jumlah testing: 1555056
Training: 2015-05-03 03:00:04.384000+00:00 - 2015-06-30 23:59:59.292000+00:00
Testing: 2015-07-01 00:00:06.967000+00:00 - 2015-09-18 02:59:47.788000+00:00


Train berisi interaksi pada periode pertama dan test berisi interaksi pada periode kedua. Data difilter menggunakan timestamp yang diubah menjadi tanggal UTC. Batas akhir tidak disertakan, sehingga dua periode yang bersebelahan tidak tumpang tindih.

In [3]:
print("Kolom events:", events.columns.tolist())
print("Kolom properties:", properties1.columns.tolist())
print("Kolom category tree:", category_tree.columns.tolist())

print("\nJenis aktivitas:")
print(events["event"].value_counts(dropna=False))

print("\nContoh timestamp:")
print(events["timestamp"].head())

Kolom events: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
Kolom properties: ['timestamp', 'itemid', 'property', 'value']
Kolom category tree: ['categoryid', 'parentid']

Jenis aktivitas:
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

Contoh timestamp:
0    1433221332117
1    1433224214164
2    1433221999827
3    1433221955914
4    1433221337106
Name: timestamp, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(r"C:\Users\ZAHRA CITRA\Downloads\archive (2)")
OUTPUT_DIR = Path.cwd() / "hasil_ujian"
OUTPUT_DIR.mkdir(exist_ok=True)

events = pd.read_csv(DATA_DIR / "events.csv")
events = events.dropna(
    subset=["timestamp", "visitorid", "event", "itemid"]
).copy()
events = events.sort_values("timestamp").reset_index(drop=True)

split_index = int(len(events) * 0.8)
split_timestamp = events.iloc[split_index]["timestamp"]

train = events[events["timestamp"] < split_timestamp].copy()
test = events[events["timestamp"] >= split_timestamp].copy()

train.to_csv(OUTPUT_DIR / "events_train.csv", index=False)
test.to_csv(OUTPUT_DIR / "events_test.csv", index=False)

print("Total data:", len(events))
print("Batas timestamp:", split_timestamp)
print("Jumlah training:", len(train))
print("Jumlah testing:", len(test))
print("Periode training:",
      train["timestamp"].min(), "-", train["timestamp"].max())
print("Periode testing:",
      test["timestamp"].min(), "-", test["timestamp"].max())

Total data: 2756101
Batas timestamp: 1439871811966
Jumlah training: 2204880
Jumlah testing: 551221
Periode training: 1430622004384 - 1439871802451
Periode testing: 1439871811966 - 1442545187788


In [22]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

event_weights = {
    "view": 1.0,
    "addtocart": 3.0,
    "transaction": 5.0
}

vsm_data = train.copy()
vsm_data["weight"] = vsm_data["event"].map(event_weights).fillna(1.0)

interactions = (
    vsm_data.groupby(["visitorid", "itemid"], as_index=False)["weight"]
    .sum()
)

visitor_ids = interactions["visitorid"].unique()
item_ids = interactions["itemid"].unique()

visitor_to_idx = {v: i for i, v in enumerate(visitor_ids)}
item_to_idx = {item: i for i, item in enumerate(item_ids)}

rows = interactions["visitorid"].map(visitor_to_idx).to_numpy()
cols = interactions["itemid"].map(item_to_idx).to_numpy()
values = interactions["weight"].to_numpy(dtype=float)

visitor_item_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(visitor_ids), len(item_ids))
)

# VSM Visitor dan VSM Item
visitor_vsm = visitor_item_matrix
item_vsm = visitor_item_matrix.T

visitor_vectors = normalize(visitor_vsm, norm="l2", axis=1)
item_vectors = normalize(item_vsm, norm="l2", axis=1)

print("Shape VSM Visitor:", visitor_vsm.shape)
print("Shape VSM Item:", item_vsm.shape)
print("Jumlah interaksi non-zero:", visitor_item_matrix.nnz)
print("Kepadatan matriks:",
      visitor_item_matrix.nnz /
      (visitor_item_matrix.shape[0] * visitor_item_matrix.shape[1]))

Shape VSM Visitor: (606763, 162894)
Shape VSM Item: (162894, 606763)
Jumlah interaksi non-zero: 925823
Kepadatan matriks: 9.367070476634013e-06


visitor_vsm.shape menunjukkan (jumlah visitor, jumlah item).
item_vsm.shape menunjukkan (jumlah item, jumlah visitor).
Matriks sparse menyimpan nilai non-zero saja, sehingga lebih hemat memori untuk data interaksi yang jarang.
visitor_vectors digunakan untuk membandingkan pola interaksi antarvisitor.
item_vectors digunakan untuk membandingkan pola pengunjung antaritem

In [21]:
import random

# Pengguna yang muncul di training dan testing
train_users = set(train["visitorid"].unique())
test_users = set(test["visitorid"].unique())

eligible_users = sorted(train_users & test_users)

if not eligible_users:
    raise ValueError(
        "Tidak ada pengguna yang muncul pada training dan testing."
    )

random.seed(42)
selected_user = random.choice(eligible_users)

print("Random seed:", 42)
print("Pengguna terpilih:", selected_user)

Random seed: 42
Pengguna terpilih: 300596


Pemilihan pengguna dilakukan secara acak menggunakan seed 42. Seed tersebut membuat proses pemilihan dapat diulang dengan hasil yang konsisten selama data dan urutan kandidat tidak berubah. Pengguna dipilih dari irisan pengguna training dan testing agar model memiliki riwayat interaksi dan rekomendasi dapat dievaluasi menggunakan data testing.

In [25]:
from sklearn.metrics.pairwise import cosine_similarity
import random

MAX_VISITORS = min(500, len(visitor_ids))
sim_matrix = cosine_similarity(visitor_vectors[:MAX_VISITORS])

random.seed(42)
reference_idx = random.randrange(MAX_VISITORS)
reference_visitor = int(visitor_ids[reference_idx])

scores = sim_matrix[reference_idx]
order = np.argsort(scores)[::-1]

similarity_rows = []

for idx in order:
    if idx == reference_idx:
        continue

    if scores[idx] <= 0:
        continue

    similarity_rows.append({
        "visitorId": reference_visitor,
        "similarVisitorId": int(visitor_ids[idx]),
        "similarity": float(scores[idx])
    })

    if len(similarity_rows) == 5:
        break

similarity_df = pd.DataFrame(similarity_rows)
display(similarity_df)

""


Untuk menghindari matriks kemiripan yang terlalu besar, contoh ini membatasi perhitungan pada maksimal 500 visitor pertama. Nilai cosine similarity semakin mendekati 1 jika dua vektor memiliki pola arah yang semakin mirip. Nilai 0 menunjukkan tidak ada kemiripan arah pada vektor non-negatif yang digunakan.

In [29]:
top3 = recommendations.head(3)

print("Tiga item rekomendasi untuk visitor:", selected_user)
display(top3)

Tiga item rekomendasi untuk visitor: 300596


,itemid,recommendation_score


In [8]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

# Bobot setiap jenis interaksi
event_weights = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5
}

train["weight"] = train["event"].map(event_weights).fillna(1)
interactions = (
    train.groupby(["visitorid", "itemid"], as_index=False)["weight"]
    .sum()
)

# Buat pemetaan ID ke indeks matriks
user_ids = interactions["visitorid"].unique()
item_ids = interactions["itemid"].unique()

user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
item_to_idx = {iid: i for i, iid in enumerate(item_ids)}

rows = interactions["visitorid"].map(user_to_idx).to_numpy()
cols = interactions["itemid"].map(item_to_idx).to_numpy()
values = interactions["weight"].to_numpy(dtype=float)

# Bentuk matriks pengguna-barang
user_item_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(item_ids))
)
user_vectors = normalize(user_item_matrix, norm="l2", axis=1)

print("Jumlah pengguna:", len(user_ids))
print("Jumlah barang:", len(item_ids))
print("Ukuran matriks:", user_item_matrix.shape)

Jumlah pengguna: 1123767
Jumlah barang: 212915
Ukuran matriks: (1123767, 212915)


In [9]:
import random

train_users = set(train["visitorid"].unique())
test_users = set(test["visitorid"].unique())

eligible_users = sorted(train_users & test_users)

if not eligible_users:
    raise ValueError(
        "Tidak ada pengguna yang muncul pada training dan testing."
    )

random.seed(42)
selected_user = random.choice(eligible_users)

print("Pengguna terpilih:", selected_user)

Pengguna terpilih: 1198666


In [11]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd

knn = NearestNeighbors(
    n_neighbors=min(6, user_item_matrix.shape[0]),
    metric="cosine",
    algorithm="brute"
)

knn.fit(user_item_matrix)

selected_idx = user_to_idx[selected_user]

distances, indices = knn.kneighbors(
    user_item_matrix[selected_idx],
    n_neighbors=min(6, user_item_matrix.shape[0])
)

neighbors = []

for distance, idx in zip(distances[0], indices[0]):
    neighbor_id = user_ids[idx]

    if neighbor_id == selected_user:
        continue

    neighbors.append({
        "visitorid": int(neighbor_id),
        "similarity": float(1 - distance)
    })

    if len(neighbors) == 5:
        break

neighbors_df = pd.DataFrame(neighbors)

print("Lima pengguna terdekat:")
display(neighbors_df)

Lima pengguna terdekat:


,visitorid,similarity
0,1256557,1.0
1,1132823,1.0
2,986553,1.0
3,735838,1.0
4,935158,1.0


In [30]:
top3 = recommendations.head(3)

print("Tiga item rekomendasi untuk visitor:", selected_user)
display(top3)

Tiga item rekomendasi untuk visitor: 300596


,itemid,recommendation_score


In [31]:
import math

positive_events = {"addtocart", "transaction"}

relevant_items = set(
    test.loc[
        (test["visitorid"] == selected_user)
        & (test["event"].isin(positive_events)),
        "itemid"
    ].astype(int)
)

recommended_items = (
    recommendations["itemid"].head(5).astype(int).tolist()
)

# Hit@5
hit_at_5 = int(
    bool(set(recommended_items) & relevant_items)
)

# DCG@5
dcg = sum(
    1 / math.log2(rank + 1)
    for rank, item in enumerate(recommended_items, start=1)
    if item in relevant_items
)

# IDCG@5
ideal_hits = min(len(relevant_items), 5)
idcg = sum(
    1 / math.log2(rank + 1)
    for rank in range(1, ideal_hits + 1)
)

ndcg_at_5 = dcg / idcg if idcg > 0 else 0.0

evaluation = pd.DataFrame([{
    "visitorid": selected_user,
    "jumlah_item_relevan": len(relevant_items),
    "Hit@5": hit_at_5,
    "NDCG@5": ndcg_at_5
}])

display(evaluation)

,visitorid,jumlah_item_relevan,Hit@5,NDCG@5
0,300596,0,0,0.0


In [38]:
import os

for folder, subfolders, files in os.walk(os.getcwd()):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(folder, file))
            
print("Folder kerja saat ini:")
print(os.getcwd())

print("\nIsi folder saat ini:")
print(os.listdir())

c:\Users\ZAHRA CITRA\Documents\hasil_ujian\events_test.csv
c:\Users\ZAHRA CITRA\Documents\hasil_ujian\events_train.csv
c:\Users\ZAHRA CITRA\Documents\hmm\hmm\sample_submission.csv
c:\Users\ZAHRA CITRA\Documents\hmm\hmm\submission.csv
c:\Users\ZAHRA CITRA\Documents\hmm\hmm\Test.csv
c:\Users\ZAHRA CITRA\Documents\hmm\hmm\Train.csv
c:\Users\ZAHRA CITRA\Documents\Pindah\TECHNOKIAWAN - rev3\Documents\data\data_janji.csv
c:\Users\ZAHRA CITRA\Documents\Pindah\TECHNOKIAWAN - rev3\Documents\data\data_promosi.csv
c:\Users\ZAHRA CITRA\Documents\Pindah\TECHNOKIAWAN - rev3\Documents\data\toyota.csv
c:\Users\ZAHRA CITRA\Documents\Pindah\TECHNOKIAWAN - SADA23E\Documents\data\data_janji.csv
c:\Users\ZAHRA CITRA\Documents\Pindah\TECHNOKIAWAN - SADA23E\Documents\data\data_promosi.csv
c:\Users\ZAHRA CITRA\Documents\Pindah\TECHNOKIAWAN - SADA23E\Documents\data\toyota.csv
c:\Users\ZAHRA CITRA\Documents\Pindah\UTS_2023E_23031554193_Salsabilla Indah Rahmawati\UTS_2023E_23031554193_Salsabilla Indah Rahmawati\

In [41]:
import pandas as pd

folder = r"C:\Users\ZAHRA CITRA\Documents\hasil_ujian"

p1 = pd.read_csv(folder + r"\events_train.csv")
p2 = pd.read_csv(folder + r"\events_test.csv")

print("Data training:", p1.shape)
print("Data testing:", p2.shape)

print("\nKolom data testing:")
print(p2.columns.tolist())

display(p2.head())

print(p2[["visitorid", "itemid"]].head())

Data training: (1201045, 5)
Data testing: (1555056, 5)

Kolom data testing:
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1435717475012,457980,view,278224,NaN
1,1435718665275,554795,view,6605,NaN
2,1435720737471,1291838,view,444688,NaN
3,1435717706116,567028,view,296749,NaN
4,1435721029856,174978,view,99274,NaN


   visitorid  itemid
0     457980  278224
1     554795    6605
2    1291838  444688
3     567028  296749
4     174978   99274


In [42]:
selected_user = int(selected_user)

relevant_items = set(
    p2.loc[
        p2["visitorid"] == selected_user,
        "itemid"
    ].dropna().astype(int)
)

print("User yang diuji:", selected_user)
print("Jumlah item relevan:", len(relevant_items))
print("Item relevan:", relevant_items)

User yang diuji: 300596
Jumlah item relevan: 1
Item relevan: {444891}


In [43]:
print("Jumlah data training:", len(p1))
print("Jumlah data testing:", len(p2))
print("User terpilih:", selected_user)

print("Kolom training:", p1.columns.tolist())
print("Kolom testing:", p2.columns.tolist())

Jumlah data training: 1201045
Jumlah data testing: 1555056
User terpilih: 300596
Kolom training: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
Kolom testing: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


In [44]:
selected_user = int(selected_user)

relevant_items = set(
    p2.loc[
        p2["visitorid"] == selected_user,
        "itemid"
    ].dropna().astype(int)
)

print("Jumlah item relevan:", len(relevant_items))
print("Item relevan:", relevant_items)

Jumlah item relevan: 1
Item relevan: {444891}


In [45]:
print(recommendations)

Empty DataFrame
Columns: [itemid, recommendation_score]
Index: []


In [46]:
import numpy as np

top5 = list(recommendations)[:5]

# Hit@5
hit5 = int(any(item in relevant_items for item in top5))

# NDCG@5
dcg = sum(
    1 / np.log2(rank + 2)
    for rank, item in enumerate(top5)
    if item in relevant_items
)

ideal_hits = min(len(relevant_items), 5)

idcg = sum(
    1 / np.log2(rank + 2)
    for rank in range(ideal_hits)
)

ndcg5 = dcg / idcg if idcg > 0 else 0.0

print("Top-5 rekomendasi:", top5)
print("Hit@5:", hit5)
print("NDCG@5:", round(ndcg5, 4))

Top-5 rekomendasi: ['itemid', 'recommendation_score']
Hit@5: 0
NDCG@5: 0.0


Interpretasi hasil:
Hit@5 = 1: setidaknya satu item rekomendasi Top-5 cocok dengan item relevan.
Hit@5 = 0: tidak ada item rekomendasi Top-5 yang cocok.
NDCG@5: mengukur kualitas urutan rekomendasi; semakin mendekati 1, semakin baik.

In [47]:
print(type(recommendations))
print(recommendations)

<class 'pandas.DataFrame'>
Empty DataFrame
Columns: [itemid, recommendation_score]
Index: []


In [49]:
print("Jumlah baris training:", len(p1))
print("Kolom training:", p1.columns.tolist())
display(p1.head())

print("User terpilih:", selected_user)

user_train = p1[p1["visitorid"] == selected_user]

print("Jumlah interaksi user di training:", len(user_train))
display(user_train.head())

Jumlah baris training: 1201045
Kolom training: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


User terpilih: 300596
Jumlah interaksi user di training: 1


,timestamp,visitorid,event,itemid,transactionid
643127,1430742489222,300596,view,294259,NaN


In [51]:
train_users = set(p1["visitorid"].dropna().astype(int))
test_users = set(p2["visitorid"].dropna().astype(int))

common_users = train_users.intersection(test_users)

print("Jumlah user di training:", len(train_users))
print("Jumlah user di testing:", len(test_users))
print("Jumlah user yang ada di keduanya:", len(common_users))

selected_user = next(iter(common_users))

print("User terpilih:", selected_user)
print("Interaksi training:",
      len(p1[p1["visitorid"] == selected_user]))
print("Interaksi testing:",
      len(p2[p2["visitorid"] == selected_user]))

Jumlah user di training: 606763
Jumlah user di testing: 834708
Jumlah user yang ada di keduanya: 33891
User terpilih: 917504
Interaksi training: 227
Interaksi testing: 129


In [52]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from itertools import combinations

# Pastikan user yang dipilih
selected_user = 917504

# Gunakan event view agar rekomendasi didasarkan pada item yang dilihat
train_views = p1[p1["event"] == "view"].copy()

# Daftar item yang pernah dilihat setiap user
user_items = (
    train_views.groupby("visitorid")["itemid"]
    .apply(lambda x: set(x.dropna().astype(int)))
)

# Hitung kemunculan pasangan item pada user yang sama
pair_counts = Counter()

for items in user_items:
    if len(items) < 2:
        continue
    for item_a, item_b in combinations(sorted(items), 2):
        pair_counts[(item_a, item_b)] += 1

# Item yang pernah dilihat user terpilih
history = user_items.get(selected_user, set())

# Hitung skor item berdasarkan kemunculan bersama
scores = defaultdict(float)

for (item_a, item_b), count in pair_counts.items():
    if item_a in history and item_b not in history:
        scores[item_b] += count
    elif item_b in history and item_a not in history:
        scores[item_a] += count

# Susun hasil rekomendasi
recommendations = pd.DataFrame(
    [
        {"itemid": item, "recommendation_score": score}
        for item, score in scores.items()
    ]
)

recommendations = recommendations.sort_values(
    "recommendation_score", ascending=False
).head(5).reset_index(drop=True)

print("User terpilih:", selected_user)
print("Jumlah item dalam riwayat:", len(history))
print("\nTop-5 rekomendasi:")
display(recommendations)

User terpilih: 917504
Jumlah item dalam riwayat: 188

Top-5 rekomendasi:


,itemid,recommendation_score
0,119736,254.0
1,234255,246.0
2,456056,198.0
3,369158,194.0
4,179601,188.0


In [53]:
# Item relevan yang benar-benar muncul pada data testing user
relevant_items = set(
    p2.loc[
        p2["visitorid"] == selected_user,
        "itemid"
    ].dropna().astype(int)
)

top5 = recommendations["itemid"].astype(int).tolist()

# Hit@5
hit5 = int(any(item in relevant_items for item in top5))

# DCG@5
dcg = sum(
    1 / np.log2(rank + 2)
    for rank, item in enumerate(top5)
    if item in relevant_items
)

# IDCG@5
ideal_hits = min(len(relevant_items), 5)
idcg = sum(
    1 / np.log2(rank + 2)
    for rank in range(ideal_hits)
)

ndcg5 = dcg / idcg if idcg > 0 else 0.0

print("User:", selected_user)
print("Jumlah item relevan:", len(relevant_items))
print("Top-5 rekomendasi:", top5)
print("Hit@5:", hit5)
print("NDCG@5:", round(ndcg5, 4))

User: 917504
Jumlah item relevan: 98
Top-5 rekomendasi: [119736, 234255, 456056, 369158, 179601]
Hit@5: 1
NDCG@5: 0.214


In [54]:
print("Jumlah rekomendasi:", len(recommendations))
display(recommendations)

Jumlah rekomendasi: 5


,itemid,recommendation_score
0,119736,254.0
1,234255,246.0
2,456056,198.0
3,369158,194.0
4,179601,188.0


In [55]:
import numpy as np

relevant_items = set(
    p2.loc[
        p2["visitorid"] == selected_user,
        "itemid"
    ].dropna().astype(int)
)

top5 = recommendations["itemid"].astype(int).tolist()

hit5 = int(any(item in relevant_items for item in top5))

dcg = sum(
    1 / np.log2(rank + 2)
    for rank, item in enumerate(top5)
    if item in relevant_items
)

ideal_hits = min(len(relevant_items), 5)

idcg = sum(
    1 / np.log2(rank + 2)
    for rank in range(ideal_hits)
)

ndcg5 = dcg / idcg if idcg > 0 else 0.0

print("User terpilih:", selected_user)
print("Jumlah item relevan:", len(relevant_items))
print("Top-5 rekomendasi:", top5)
print("Hit@5:", hit5)
print("NDCG@5:", round(ndcg5, 4))

User terpilih: 917504
Jumlah item relevan: 98
Top-5 rekomendasi: [119736, 234255, 456056, 369158, 179601]
Hit@5: 1
NDCG@5: 0.214
